In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflowjs as tfjs
import numpy as np
import struct

In [ ]:
# DOWNLOAD EMNIST DATASET FROM https://www.nist.gov/itl/products-and-services/emnist-dataset

def loadLocalDataset(file_path):
    with open(file_path, 'rb') as f:
        zero, data_type, dims = struct.unpack('>HBB', f.read(4))
        shape = tuple(struct.unpack('>I', f.read(4))[0] for d in range(dims))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

def loadEmnist():
    train_images = "./emnist-letters-train-images-idx3-ubyte"
    train_labels = "./emnist-letters-train-labels-idx1-ubyte"
    test_images = "./emnist-letters-test-images-idx3-ubyte"
    test_labels = "./emnist-letters-test-labels-idx1-ubyte"

    x_train = loadLocalDataset(train_images)
    y_train = loadLocalDataset(train_labels) - 1  # Subtract 1 to make labels zero-based
    x_test = loadLocalDataset(test_images)
    y_test = loadLocalDataset(test_labels) - 1  # Subtract 1 to make labels zero-based

    return x_train, y_train, x_test, y_test

In [ ]:
# Load data
x_train, y_train, x_test, y_test = loadEmnist()

# Print dataset details
print(f"Training images: {x_train.shape[0]} images of size {x_train.shape[1]}x{x_train.shape[2]}")
print(f"Training labels: {y_train.shape[0]} labels as {np.unique(y_train)} (zero-based numerical mapping for A-Z)")
print(f"Test images: {x_test.shape[0]} images of size {x_test.shape[1]}x{x_test.shape[2]}")
print(f"Test labels: {y_test.shape[0]} labels as {np.unique(y_test)} (zero-based numerical mapping for A-Z)")

In [ ]:


# Transpose images by swapping height and width dimensions (normally 0, 1, 2)
x_train = x_train.transpose(0, 2, 1)
x_test = x_test.transpose(0, 2, 1)

plt.imshow(x_train[0], cmap='gray')
plt.show()

In [ ]:
# Normalize data
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

In [ ]:
# Define the CNN model
model = tf.keras.models.Sequential([
    # First convolutional layer
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    # Second convolutional layer
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    # Third convolutional layer
    tf.keras.layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    # Fourth convolutional layer
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    # Fifth convolutional layer
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    # Sixth convolutional layer
    tf.keras.layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    # Final fully connected Dense layer
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(26, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:

# Train the model
model.fit(x_train, y_train, epochs=30)

# Evaluate the model
val_loss, val_acc = model.evaluate(x_test, y_test)
print(val_loss, val_acc)

In [ ]:
# USE PYTHON 3.8 FOR CORRECT WEBCONVERSION (model.js + group1-shard1of1.bin)

# Save model for TensorFlow.js
tfjs.converters.save_keras_model(model, 'model')